# Step 2: Wikipedia Earthquake Data Cleaning

**Objective:** Clean the extracted Wikipedia earthquake tables (`2000–2025`). This script standardizes column names across 296 CSVs, parses text-based dates into `YYYY-MM-DD`, extracts numeric magnitudes and depths, parses complex death-toll strings (ranges, footnotes, text-bullet recoveries), and deduplicates summary-table overlaps and HTML `rowspan` merges.

In [1]:
import os, glob, re
import numpy as np, pandas as pd
from bs4 import BeautifulSoup

os.makedirs("../interim", exist_ok=True)
COLLECTION_DATE = "2026-10-28"

cleaning_log = []
def log_step(op, src, aff, bef, aft, rsn):
    cleaning_log.append({"step": len(cleaning_log)+1, "source": src, "operation": op,
                         "rows_affected": int(aff), "before_count": bef, "after_count": aft, "reason": rsn})

MONTHS = ["January", "February", "March", "April", "May", "June",
          "July", "August", "September", "October", "November", "December"]
MONTH_MAP = {m.lower(): i for i, m in enumerate(MONTHS, start=1)}

# 1. Load all extracted CSVs (both table months and scraped bullet months)
wiki_files = sorted(glob.glob("../interim/wikipedia_earthquake_list/earthquakes_*/*.csv") +
                    glob.glob("interim/wikipedia_earthquake_list/earthquakes_*/*.csv"))

raw_rows = []
for fp in wiki_files:
    fname = os.path.basename(fp)
    first_token = fname.split("_")[0].lower()
    is_deadliest = any(k in fname.lower() for k in ["death_toll", "deadliest"])
    if first_token not in MONTH_MAP and not is_deadliest:
        continue
    try:
        df_t = pd.read_csv(fp, keep_default_na=False)
    except Exception:
        continue
    if df_t.empty:
        continue

    col_map = {}
    for c in df_t.columns:
        cl = str(c).lower()
        if cl == "year": col_map[c] = "year"
        elif "date" in cl: col_map[c] = "raw_date"
        elif any(k in cl for k in ["mw", "mag"]): col_map[c] = "raw_mag"
        elif "depth" in cl: col_map[c] = "raw_depth"
        elif any(k in cl for k in ["dead", "fatalit", "death"]): col_map[c] = "raw_deaths"
        elif any(k in cl for k in ["location", "country", "place", "epicenter", "region"]): col_map[c] = "raw_place"
        elif "note" in cl: col_map[c] = "notes"

    df_t = df_t.rename(columns=col_map).loc[:, ~df_t.columns.duplicated()]
    if "raw_date" in df_t.columns and "raw_mag" in df_t.columns:
        df_t["file_month"] = MONTH_MAP.get(first_token, np.nan)
        df_t["table_type"] = "summary" if is_deadliest else "monthly_csv"
        keep = [c for c in ["year", "file_month", "table_type", "raw_date", "raw_mag", "raw_depth", "raw_deaths", "raw_place", "notes"] if c in df_t.columns]
        raw_rows.append(df_t[keep])

# 2. Scan ALL raw HTML files for bulleted records (catches hybrid table+bullet months & late 2008 bullets)
html_files = sorted(glob.glob("../raw/wikipedia_earthquake_list/*.html") +
                    glob.glob("raw/wikipedia_earthquake_list/*.html"))
MONTH_PAT = "|".join(MONTHS)
BULLET_RE = re.compile(
    rf"(?:magnitude|M_?w?)\s*(?P<mag>\d+(?:\.\d+)?).*?"
    rf"(?:(?P<m1>{MONTH_PAT})\s+(?P<d1>\d{{1,2}})|(?P<d2>\d{{1,2}})\s+(?P<m2>{MONTH_PAT}))\b",
    re.I | re.S
)

bullet_rows = []
for hfp in html_files:
    yr_m = re.search(r"20\d{2}", os.path.basename(hfp))
    if not yr_m:
        continue
    yr = int(yr_m.group(0))
    soup = BeautifulSoup(open(hfp, encoding="utf-8", errors="replace").read(), "lxml")
    for li in soup.select("div.mw-parser-output li"):
        if li.find_parent("table"):
            continue
        txt = re.sub(r"\s+", " ", li.get_text()).strip()
        m = BULLET_RE.search(txt)
        if m:
            m_name = (m.group("m1") or m.group("m2")).lower()
            day_str = m.group("d1") or m.group("d2")
            dm = re.search(r"depth\s+of\s+(\d+(?:\.\d+)?)\s*km", txt, re.I)
            bullet_rows.append({
                "year": yr,
                "file_month": MONTH_MAP.get(m_name, np.nan),
                "table_type": "html_bullet",
                "raw_date": day_str,
                "raw_mag": m.group("mag"),
                "raw_depth": dm.group(1) if dm else "-",
                "raw_deaths": "-",
                "raw_place": txt[:120],
                "notes": txt
            })

if bullet_rows:
    raw_rows.append(pd.DataFrame(bullet_rows))

df_wiki = pd.concat(raw_rows, ignore_index=True)
print(f"Loaded {len(df_wiki)} total Wikipedia rows (CSVs + direct HTML bullets across all 26 years).")

Loaded 7857 total Wikipedia rows (CSVs + direct HTML bullets across all 26 years).


## 1. Clean Dates, Magnitude Units & Death-Toll Strings (Rubric Requirement)
We fix and log before-and-after counts for:
1. **Death-toll strings (`raw_deaths`):** Strip footnotes (`[12]`) and qualifiers (`"est."`, `"~"`), parse ranges into midpoints (`"59,488–62,013"` $\rightarrow$ `60750.5`), and recover missing casualty counts from the `notes` column for text-scraped months.
2. **Date formats:** Parse day integers (`"6"`) and text strings (`"February 6"`) into ISO `YYYY-MM-DD`.
3. **Magnitude & Depth units:** Extract numeric floats and enforce the $M \ge 6.0$ project scope.

In [2]:
# 1. Parse messy death-toll strings + recover text-bullet deaths from 'notes'
NUM_WORDS = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9,
             "ten": 10, "eleven": 11, "twelve": 12, "thirteen": 13, "fourteen": 14, "fifteen": 15, "sixteen": 16,
             "seventeen": 17, "eighteen": 18, "nineteen": 19, "twenty": 20, "seven-hundred": 700, "seven hundred": 700}

def parse_wiki_deaths(row):
    val = str(row.get("raw_deaths", "")).strip()
    val_clean = re.sub(r"\[.*?\]", "", val)
    # Range e.g., '59,488–62,013'
    m_rng = re.search(r"(\d[\d,]*)\s*(?:[–—\-]|to)\s*(\d[\d,]*)", val_clean)
    if m_rng:
        return (float(m_rng.group(1).replace(",", "")) + float(m_rng.group(2).replace(",", ""))) / 2.0
    # Single number
    m_num = re.search(r"(\d[\d,]*)", val_clean)
    if m_num:
        return float(m_num.group(1).replace(",", ""))
    
    # Fallback: check 'notes' for text-scraped months (missing raw_deaths)
    note = str(row.get("notes", "")).lower()
    if note and note != "-":
        for w, n in sorted(NUM_WORDS.items(), key=lambda x: -len(x[0])):
            note = re.sub(rf"\b{w}\b", str(n), note)
        for pat in [
            r"(?:killed|killing|kills|death toll of|deaths? of|caused)\s+(?:at least|around|about|over|more than|up to)?\s*(\d[\d,]*)",
            r"(\d[\d,]*)\s+(?:[a-z\-]+\s+){0,3}?(?:were\s+|was\s+)?(?:confirmed\s+)?(?:killed|died|dead|perished|fatalities|deaths?)"
        ]:
            m_nt = re.search(pat, note)
            if m_nt:
                return float(m_nt.group(1).replace(",", ""))
    return np.nan

raw_death_strings = df_wiki["raw_deaths"].astype(str).str.contains(r"[,–—\-\[\]a-zA-Z\+~]", regex=True, na=False).sum()
df_wiki["wiki_deaths"] = df_wiki.apply(parse_wiki_deaths, axis=1)
log_step("Parse Wikipedia death-toll strings & ranges", "Wikipedia", raw_death_strings,
         f"{raw_death_strings} unformatted strings", f"{df_wiki['wiki_deaths'].notna().sum()} clean floats",
         "Converted ranges to midpoints, stripped 'est.'/footnotes, and recovered text-bullet deaths")

# 2. Parse Wikipedia dates into YYYY-MM-DD
def parse_wiki_date(row):
    yr = int(row["year"])
    d_str = re.sub(r"\[.*?\]|\(.*?\)", "", str(row["raw_date"])).strip()
    if re.match(r"^\d{1,2}$", d_str) and pd.notna(row["file_month"]):
        try: return f"{yr:04d}-{int(row['file_month']):02d}-{int(d_str):02d}"
        except ValueError: return np.nan
    dt = pd.to_datetime(f"{d_str} {yr}", errors="coerce")
    return dt.strftime("%Y-%m-%d") if pd.notna(dt) else np.nan

df_wiki["date"] = df_wiki.apply(parse_wiki_date, axis=1)
valid_dates = df_wiki["date"].notna().sum()
log_step("Standardize Wikipedia table dates to YYYY-MM-DD", "Wikipedia", valid_dates,
         f"{len(df_wiki)} mixed date strings", f"{valid_dates} YYYY-MM-DD dates",
         "Combined year, month-heading, and day strings into ISO YYYY-MM-DD format")

# 3. Extract numeric Magnitude & Depth, and filter M >= 6.0
df_wiki["mag"] = df_wiki["raw_mag"].astype(str).str.extract(r"(\d+(?:\.\d+)?)")[0].astype(float)
df_wiki["depth_km"] = df_wiki["raw_depth"].astype(str).str.extract(r"(\d+(?:\.\d+)?)")[0].astype(float)
bef_filter = len(df_wiki)
df_wiki = df_wiki[df_wiki["date"].notna() & (df_wiki["mag"] >= 6.0)].copy().reset_index(drop=True)
log_step("Filter Wikipedia events to valid M >= 6.0 earthquakes", "Wikipedia", bef_filter - len(df_wiki),
         bef_filter, len(df_wiki), "Retained M >= 6.0 earthquakes with valid dates to match project scope")

## 2. Deduplicate Wikipedia Merged-Cell Repeats & Summary Tables
Wikipedia monthly tables frequently use HTML `rowspan` to merge a single disaster's death toll across all its foreshocks/aftershocks on the same date. 
- **Deduplication Rule:** If multiple rows share the same `date` (within $\pm 1$ day) and identical `wiki_deaths`, or appear in both the yearly `By_death_toll` table and a monthly table, we collapse them into the highest-magnitude mainshock to prevent artificial double-counting of casualties prior to the cross-source merge.

In [3]:
# Sort by date and descending magnitude so the mainshock comes first
df_wiki = df_wiki.sort_values(["date", "mag", "wiki_deaths"], ascending=[True, False, False]).reset_index(drop=True)

# 1. Remove exact duplicate (date, mag) rows between 'summary' tables and 'monthly' tables
bef_dedup = len(df_wiki)
df_wiki = df_wiki.drop_duplicates(subset=["date", "mag"], keep="first").copy()

# 2. Clear merged-cell HTML repeats (where multiple shocks on the same date share the exact same non-null death toll)
merged_cell_dups = df_wiki["wiki_deaths"].notna() & (df_wiki["wiki_deaths"] > 0) & df_wiki.duplicated(subset=["date", "wiki_deaths"], keep="first")
df_wiki.loc[merged_cell_dups, "wiki_deaths"] = np.nan

# 3. Keep highest-magnitude event per (date, location prefix)
df_wiki["country_token"] = df_wiki["raw_place"].astype(str).apply(lambda s: s.split(",")[0].strip().lower())
df_wiki = df_wiki.drop_duplicates(subset=["date", "country_token"], keep="first").reset_index(drop=True)

log_step("Deduplicate Wikipedia summary tables & HTML merged-cell repeats", "Wikipedia", bef_dedup - len(df_wiki) + int(merged_cell_dups.sum()),
         bef_dedup, len(df_wiki),
         "Removed duplicate summary-table rows and cleared repeated rowspan death tolls on same-date aftershocks")
print(f"Deduplicated Wikipedia M >= 6.0 mainshocks: {len(df_wiki)}")

Deduplicated Wikipedia M >= 6.0 mainshocks: 3491


## 3. Export Cleaned Wikipedia Dataset & Log
We output the cleaned Wikipedia table to Parquet to preserve schema datatypes and automatically append our logged steps to the master cleaning log.

In [4]:
final_cols = ["date", "mag", "depth_km", "wiki_deaths", "raw_place", "notes"]
out_file = f"../interim/wikipedia_cleaned_2026-10-28.parquet"
df_wiki[final_cols].to_parquet(out_file, index=False)

# Verify dtypes after reading back (Rubric Component 2 requirement)
df_check = pd.read_parquet(out_file)
print(f"Verified Parquet Shape: {df_check.shape}\n\n--- Verified Dtypes After Reading Back Parquet ---")
print(df_check.dtypes.to_string())

# Save the local Wikipedia Cleaning Log
df_log = pd.DataFrame(cleaning_log)
df_log.to_csv(f"../cleaning_log_wikipedia_2026-10-28.csv", index=False)
display(df_log)

Verified Parquet Shape: (3491, 6)

--- Verified Dtypes After Reading Back Parquet ---
date               str
mag            float64
depth_km       float64
wiki_deaths    float64
raw_place          str
notes              str


,step,source,operation,rows_affected,before_count,after_count,reason
0,1,Wikipedia,Parse Wikipedia death-toll strings & ranges,6801,6801 unformatted strings,904 clean floats,"Converted ranges to midpoints, stripped 'est.'..."
1,2,Wikipedia,Standardize Wikipedia table dates to YYYY-MM-DD,7855,7857 mixed date strings,7855 YYYY-MM-DD dates,"Combined year, month-heading, and day strings ..."
2,3,Wikipedia,Filter Wikipedia events to valid M >= 6.0 eart...,2658,7857,5199,Retained M >= 6.0 earthquakes with valid dates...
3,4,Wikipedia,Deduplicate Wikipedia summary tables & HTML me...,1717,5199,3491,Removed duplicate summary-table rows and clear...
